In [ ]:
# Cell 1 — Install all packages
!pip install -q \
    "numpy>=1.22,<2.1" \
    "langchain==0.3.25" \
    "langchain-core==0.3.62" \
    "langchain-text-splitters==0.3.8" \
    "langchain-google-genai==2.1.5" \
    "langchain-chroma==0.2.4" \
    "langchain-huggingface==0.1.2" \
    "chromadb>=1.0.9" \
    "rank-bm25==0.2.2" \
    "sentence-transformers>=2.6.0" \
    "gradio>=4.40,<6.0" \
    "gitpython>=3.1"

# Cell 2 — Restart kernel so all packages are visible to Python
# import os
# os.kill(os.getpid(), 9)

In [ ]:
## Setup api keys
import os

# Suppress ChromaDB telemetry warnings (harmless posthog SDK incompatibility)
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY_IMPL"] = "none"

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab secrets ✓")
except Exception:
    if "GOOGLE_API_KEY" not in os.environ:
        from getpass import getpass
        os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")
    print("API key set ✓")

API key loaded from Colab secrets ✓


In [ ]:
#Import codebases from git
import shutil
from pathlib import Path
import git

# Change this to point at your own repo any time.
REPO_URL = "https://github.com/Priyabrata111/Ecommerce-App"
LOCAL_PATH = Path("/tmp/repo")

# Wipe any previous clone
if LOCAL_PATH.exists():
    shutil.rmtree(LOCAL_PATH)

git.Repo.clone_from(REPO_URL, LOCAL_PATH, depth=1)
print(f"Cloned into {LOCAL_PATH}")
print(f"Files: {sum(1 for _ in LOCAL_PATH.rglob('*') if _.is_file())}")

Cloned into /tmp/repo
Files: 107


In [ ]:
# Show the directory tree (limit depth so output is readable)
def tree(p: Path, prefix: str = "", depth: int = 2) -> None:
    if depth < 0:
        return
    entries = sorted([e for e in p.iterdir() if not e.name.startswith(".")])
    for i, entry in enumerate(entries):
        is_last = i == len(entries) - 1
        connector = "└── " if is_last else "├── "
        print(prefix + connector + entry.name)
        if entry.is_dir():
            extension = "    " if is_last else "│   "
            tree(entry, prefix + extension, depth - 1)

tree(LOCAL_PATH)

├── client
│   └── build
│       ├── asset-manifest.json
│       ├── favicon.ico
│       ├── images
│       ├── index.html
│       ├── logo192.png
│       ├── logo512.png
│       ├── manifest.json
│       ├── robots.txt
│       └── static
├── config
│   └── db.js
├── controllers
│   ├── authController.js
│   ├── categoryController.js
│   └── productController.js
├── helpers
│   └── authHelper.js
├── middlewares
│   └── authMiddleware.js
├── models
│   ├── categoryModel.js
│   ├── orderModel.js
│   ├── productModel.js
│   └── userModel.js
├── package-lock.json
├── package.json
├── routes
│   ├── authRoute.js
│   ├── categoryRoutes.js
│   └── productRoutes.js
└── server.js


In [ ]:


from pathlib import Path
from typing import List
from langchain_core.documents import Document
# Extensions relevant for MERN / Node.js / React
CODE_EXTENSIONS = {
    ".js",
    ".jsx",
    ".json",
    ".md",
    ".css",
    ".html",
    ".env",
}

# Folders to ignore
SKIP_DIRS = {
    ".git",
    "__pycache__",
    "node_modules",
    "dist",
    ".next",
    ".cache",
}
# Function to load the git repo as langchain model

def load_repo_documents(repo_path: Path) -> List[Document]:
    docs = []

    for path in repo_path.rglob("*"):

        # Skip directories
        if not path.is_file():
            continue

        # Skip unwanted folders
        if any(part in SKIP_DIRS for part in path.parts):
            continue

        # Skip unsupported file types
        if path.suffix.lower() not in CODE_EXTENSIONS:
            continue

        try:
            text = path.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            continue

        rel = path.relative_to(repo_path)

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": str(rel),
                    "extension": path.suffix,
                },
            )
        )

    return docs


# Function call to load the repo as langchain documents
raw_docs = load_repo_documents(LOCAL_PATH)

print(f"Loaded {len(raw_docs)} files")

for d in raw_docs[:5]:
    print(
        f"{d.metadata['source']:40s} "
        f"({len(d.page_content)} chars)"
    )

Loaded 22 files
package-lock.json                        (135391 chars)
server.js                                (1440 chars)
package.json                             (854 chars)
helpers/authHelper.js                    (384 chars)
middlewares/authMiddleware.js            (829 chars)


In [ ]:
#facing issue with numpy version for langchain

# !pip install numpy==1.26.4
# !pip install -U langchain-text-splitters

In [ ]:
#Restart automatically as Python already loaded old NumPy binary into memory
# import os
# os.kill(os.getpid(), 9)

In [ ]:
# !pip install langchain-text-splitters spacy==3.7.2 thinc==8.2.2

In [ ]:
# !pip install -q "langchain-text-splitters==0.3.8"

In [ ]:
## break the files into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def dedup_chunks(chunks: List[Document]) -> List[Document]:
    """Drop chunks with identical content. Keeps the first occurrence per (source, content) pair."""
    seen = set()
    out = []
    for c in chunks:
        key = (c.metadata.get("source", ""), c.page_content)
        if key not in seen:
            seen.add(key)
            out.append(c)
    return out


naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""],
)

naive_chunks = dedup_chunks(naive_splitter.split_documents(raw_docs))
print(f"Created {len(naive_chunks)} chunks from {len(raw_docs)} files (after dedup)")
print(f"Average chunk size: {sum(len(c.page_content) for c in naive_chunks) // len(naive_chunks)} chars")
print()
print("--- Sample chunk ---")
print(f"From: {naive_chunks[10].metadata['source']}")
print(naive_chunks[10].page_content[:400])

Created 1704 chunks from 22 files (after dedup)
Average chunk size: 894 chars

--- Sample chunk ---
From: package-lock.json
"@aws-sdk/util-endpoints": "3.254.0",
        "@aws-sdk/util-retry": "3.254.0",
        "@aws-sdk/util-user-agent-browser": "3.254.0",
        "@aws-sdk/util-user-agent-node": "3.254.0",
        "@aws-sdk/util-utf8-browser": "3.188.0",
        "@aws-sdk/util-utf8-node": "3.208.0",
        "tslib": "^2.3.1"
      },
      "engines": {
        "node": ">=14.0.0"
      }
    },
    "node_modules/@aws


In [ ]:
#Sample check

import random
random.seed(42)
samples = random.sample(naive_chunks, min(3, len(naive_chunks)))

for i, c in enumerate(samples, 1):
    print(f"─── Sample {i} of 3 ───")
    print(f"Source: {c.metadata['source']}")
    print(f"Length: {len(c.page_content)} chars")
    print(f"Content (first 250 chars):")
    print(c.page_content[:250])
    print()

─── Sample 1 of 3 ───
Source: client/build/static/js/main.0def76c2.js
Length: 965 chars
Content (first 250 chars):
Zs=Ds.isStandardBrowserEnv?function(){var e,t=/(msie|trident)/i.test(navigator.userAgent),n=document.createElement("a");function r(e){var r=e;return t&&(n.setAttribute("href",r),r=n.href),n.setAttribute("href",r),{href:n.href,protocol:n.protocol?n.pr

─── Sample 2 of 3 ───
Source: client/build/static/js/main.0def76c2.js
Length: 997 chars
Content (first 250 chars):
0,n.setAttributes=r},{}],47:[function(e,t,n){"use strict";function r(){return"xxxxxxxx-xxxx-4xxx-yxxx-xxxxxxxxxxxx".replace(/[xy]/g,(function(e){var t=16*Math.random()|0;return("x"===e?t:3&t|8).toString(16)}))}t.exports=r},{}],48:[function(e,t,n){"us

─── Sample 3 of 3 ───
Source: package-lock.json
Length: 980 chars
Content (first 250 chars):
"version": "7.3.8",
      "resolved": "https://registry.npmjs.org/semver/-/semver-7.3.8.tgz",
      "integrity": "sha512-NB1ctGL5rlHrPJtFDVIVzTyQylMLu9N9VICA6HSFJo8MCGVTMW6g

In [ ]:
# pip install -U langchain-huggingface sentence-transformers chromadb

In [ ]:
import numpy as np
print(np.__version__)  # should be 1.x.x

2.0.2


In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Local embedding model
# Downloads once, then runs locally
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Quick smoke test
v = embeddings.embed_query("smoke test")
print(f"✓ Local embeddings loaded (vector dim = {len(v)})")

# Create Chroma vector database
vectorstore_v1 = Chroma.from_documents(
    documents=naive_chunks,
    embedding=embeddings,
    collection_name="codebase_v1",
    persist_directory="./chroma_db",  # optional but recommended
)

print(f"✓ Indexed {len(naive_chunks)} chunks")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Local embeddings loaded (vector dim = 384)
✓ Indexed 1704 chunks


In [ ]:
sample_text = "How does authentication work in this codebase?"
sample_vec = embeddings.embed_query(sample_text)

print(f"Query:           {sample_text!r}")
print(f"Vector length:   {len(sample_vec)}")
print(f"First 10 dims:   {[round(x, 4) for x in sample_vec[:10]]}")
print(f"L2 norm:         {sum(x*x for x in sample_vec) ** 0.5:.4f}")
print()
print("This is what gets stored in Chroma — one of these for every chunk.")
print("Similarity search compares the query vector to all chunk vectors via cosine distance.")

Query:           'How does authentication work in this codebase?'
Vector length:   384
First 10 dims:   [-0.077, -0.0142, -0.0617, -0.0495, -0.0297, -0.0008, 0.0978, -0.0071, 0.009, -0.0135]
L2 norm:         1.0000

This is what gets stored in Chroma — one of these for every chunk.
Similarity search compares the query vector to all chunk vectors via cosine distance.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

prompt = ChatPromptTemplate.from_template('''You are a helpful coding assistant answering questions about a codebase.
Use ONLY the context below. If the answer isn\'t in the context, say "I don\'t see that in the code."

Context:
{context}

Question: {question}

Answer:''')


def format_context(docs):
    return "\n\n---\n\n".join(
        f"# File: {d.metadata['source']}\n{d.page_content}" for d in docs
    )


def show_retrieval(question, docs):
    """Print which chunks were retrieved. Call before generation so students see retrieval as a step."""
    print(f"📚 Retrieved {len(docs)} chunks for: {question!r}")
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "?")
        size = len(d.page_content)
        line_info = ""
        if "start_line" in d.metadata:
            line_info = f":{d.metadata['start_line']}-{d.metadata['end_line']}"
        print(f"  {i}. {src}{line_info} ({size} chars)")
    print()


def ask_v1(question: str, k: int = 4) -> str:
    docs = vectorstore_v1.similarity_search(question, k=k)
    show_retrieval(question, docs)
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"context": format_context(docs), "question": question})


print(ask_v1("How does authentication work in this codebase?"))

📚 Retrieved 4 chunks for: 'How does authentication work in this codebase?'
  1. client/build/static/js/main.0def76c2.js (982 chars)
  2. client/build/static/js/main.0def76c2.js (982 chars)
  3. client/build/static/js/main.0def76c2.js (951 chars)
  4. client/build/static/js/main.0def76c2.js (951 chars)

The codebase includes a "forgot password" mechanism. It sends a POST request to `/api/v1/auth/forgot-password` with the user's email, new password, and an answer. If successful, it displays a success message and redirects the user to `/login`. If there's an error, it displays an error message.


In [ ]:
print("Q3: Multi-file question")
print("-" * 60)
print(ask_v1("What happens when a user clicks on Buy button?"))

Q3: Multi-file question
------------------------------------------------------------
📚 Retrieved 4 chunks for: 'What happens when a user clicks on Buy button?'
  1. client/build/static/js/main.0def76c2.js (975 chars)
  2. client/build/static/js/main.0def76c2.js (975 chars)
  3. client/build/static/js/main.0def76c2.js (924 chars)
  4. client/build/static/js/main.0def76c2.js (924 chars)

When the `apple-pay-button` is clicked, the `_showPaymentSheet` function is called. This function sets `_sessionInProgress` to true and creates a payment request using `this.applePayInstance.createPaymentRequest`.


Above answers were generated using 1000-character chunking. Some answers were not accurate, so the chunking strategy will be modified to semantic chunking to provide more accurate results.